# Minimal LangGraph Architecture Example
This notebook demonstrates a simple orchestrator -> classifier -> specialized agent -> ReAct -> tool flow.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    query:str
    intent:str
    response:str


In [ ]:
def classify_intent(state):
    q=state["query"].lower()
    if "claim" in q:
        state["intent"]="claims"
    elif "benefit" in q:
        state["intent"]="benefits"
    else:
        state["intent"]="case"
    return state

def claims_agent(state):
    state["response"]="Claims Agent Selected"
    return state

def benefits_agent(state):
    state["response"]="Benefits Agent Selected"
    return state

def case_agent(state):
    state["response"]="Case Management Agent Selected"
    return state

def claim_tool(query):
    return "Claim Status: Approved"

def react_agent(state):
    state["response"]=claim_tool(state["query"])
    return state

def route(state):
    return state["intent"]


In [ ]:
graph=StateGraph(AgentState)
graph.add_node("classifier", classify_intent)
graph.add_node("claims", claims_agent)
graph.add_node("benefits", benefits_agent)
graph.add_node("case", case_agent)
graph.add_node("react", react_agent)
graph.set_entry_point("classifier")
graph.add_conditional_edges("classifier", route,
                            {"claims":"claims","benefits":"benefits","case":"case"})
graph.add_edge("claims","react")
graph.add_edge("benefits","react")
graph.add_edge("case","react")
graph.add_edge("react", END)
app=graph.compile()

In [ ]:
result=app.invoke({"query":"What is my claim status?"})
print(result)